# AML ML Preparation — EDA, Feature Selection & Baseline Model
---
**Target**: `is_aml` (binary)

**Feature Strategy:**
- **RETAIN AS-IS**: All 126 rule flags + encoded categoricals (no selection applied)
- **SELECT FROM**: Graph/velocity/balance features only (correlation + VIF + importance)
- **EXCLUDE**: Typology signal, convergence risk, temporal risk (leakage from typology labels)
- **EXCLUDE**: FIS, alert_level, fis_band (post-hoc derived scores)

**Class Imbalance**: SMOTE, class weights, and threshold tuning compared


## 1 — Environment Setup


In [1]:
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings("ignore")
from collections import defaultdict
from datetime import datetime

# Viz
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (14, 6)

OUTPUT_DIR = "ml_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Environment ready")



Environment ready


## 2 — Load Feature-Engineered Data


In [2]:
INPUT_FILE = "../outputs_updated/stg_transactions_features_V2.parquet"

if not os.path.exists(INPUT_FILE):
    # Try alternate paths
    for alt in ["stg_transactions_features_V2.parquet",
                "../aml_features_output/stg_transactions_features_V2.parquet",
                "../stg_transactions_features_V2.parquet"]:
        if os.path.exists(alt):
            INPUT_FILE = alt
            break

df = pd.read_parquet(INPUT_FILE)
print(f"Loaded: {INPUT_FILE}")
print(f"  {len(df):,} rows × {len(df.columns)} columns")
print(f"  is_aml distribution: 0={( df['is_aml']==0).sum():,}  1={(df['is_aml']==1).sum():,}  ({(df['is_aml']==1).mean()*100:.1f}%)")



Loaded: ../outputs_updated/stg_transactions_features_V2.parquet
  333,875 rows × 318 columns
  is_aml distribution: 0=204,230  1=129,645  (38.8%)


## 3 — Initial EDA: Target Distribution & Data Quality


In [3]:
print("=" * 90)
print("INITIAL EDA")
print("=" * 90)

# ── 3.1: Target distribution ──
print("\n── 3.1: Target Variable (is_aml) ──")
target_counts = df["is_aml"].value_counts().sort_index()
for val, cnt in target_counts.items():
    print(f"  is_aml={val}: {cnt:>10,} ({cnt/len(df)*100:.1f}%)")
imbalance_ratio = target_counts[0] / max(target_counts[1], 1)
print(f"  Imbalance ratio: {imbalance_ratio:.1f}:1 (Clean:AML)")

# ── 3.2: Column type breakdown ──
print("\n── 3.2: Column Types ──")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
object_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
print(f"  Numeric: {len(numeric_cols)} | Object/String: {len(object_cols)} | Boolean: {len(bool_cols)}")

# ── 3.3: Missing values ──
print("\n── 3.3: Missing Values (top 20) ──")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).sort_values(ascending=False)
missing_top = missing_pct[missing_pct > 0].head(20)
if len(missing_top) > 0:
    for col, pct in missing_top.items():
        print(f"  {col:<50s} {pct:>6.2f}%")
else:
    print("  No missing values found")

# ── 3.4: Typology distribution within AML ──
print("\n── 3.4: Typology Distribution (within is_aml=1) ──")
if "aml_typology" in df.columns:
    aml_df = df[df["is_aml"] == 1]
    all_typs = {}
    for t in aml_df["aml_typology"].dropna():
        for part in str(t).split("; "):
            part = part.strip()
            if part: all_typs[part] = all_typs.get(part, 0) + 1
    for typ, cnt in sorted(all_typs.items(), key=lambda x: -x[1]):
        print(f"  {typ:<40s} {cnt:>8,} ({cnt/len(aml_df)*100:.1f}%)")

# ── 3.5: Key numeric feature statistics ──
print("\n── 3.5: Key Feature Statistics (AML vs Clean) ──")
key_features = ["transaction_amount", "rule_score", "fraud_intensity_score",
                "sender_acct_txn_count_24h", "sender_acct_outflow_amt_24h",
                "sender_acct_unique_counterparties_7d", "ip_risk_score"]

existing_keys = [f for f in key_features if f in df.columns]
print(f"\n  {'Feature':<45s} │ {'AML Mean':>10s} {'Clean Mean':>10s} {'Ratio':>7s} │ {'AML Med':>10s} {'Clean Med':>10s}")
print("  " + "─" * 100)
for feat in existing_keys:
    am = df.loc[df["is_aml"]==1, feat].mean()
    cm = df.loc[df["is_aml"]==0, feat].mean()
    amed = df.loc[df["is_aml"]==1, feat].median()
    cmed = df.loc[df["is_aml"]==0, feat].median()
    ratio = am / max(cm, 0.0001)
    print(f"  {feat:<45s} │ {am:>10.2f} {cm:>10.2f} {ratio:>6.2f}x │ {amed:>10.2f} {cmed:>10.2f}")

# ── 3.6: Plots ──
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Initial EDA — Target & Key Features", fontsize=14, fontweight="bold")

# Target bar
ax = axes[0, 0]
colors = ["#2ecc71", "#e74c3c"]
target_counts.plot(kind="bar", ax=ax, color=colors)
ax.set_title("Target Distribution (is_aml)")
ax.set_xticklabels(["Clean (0)", "AML (1)"], rotation=0)
for i, v in enumerate(target_counts):
    ax.text(i, v + len(df)*0.01, f"{v:,}\n({v/len(df)*100:.1f}%)", ha="center", fontsize=9)

# FIS by AML status
ax = axes[0, 1]
if "fraud_intensity_score" in df.columns:
    df.loc[df["is_aml"]==0, "fraud_intensity_score"].hist(bins=50, alpha=0.6, ax=ax, label="Clean", color="#2ecc71", density=True)
    df.loc[df["is_aml"]==1, "fraud_intensity_score"].hist(bins=50, alpha=0.6, ax=ax, label="AML", color="#e74c3c", density=True)
    ax.set_title("FIS Distribution: Clean vs AML")
    ax.legend()

# Rule score by AML
ax = axes[1, 0]
if "rule_score" in df.columns:
    df.loc[df["is_aml"]==0, "rule_score"].hist(bins=50, alpha=0.6, ax=ax, label="Clean", color="#2ecc71", density=True)
    df.loc[df["is_aml"]==1, "rule_score"].hist(bins=50, alpha=0.6, ax=ax, label="AML", color="#e74c3c", density=True)
    ax.set_title("Rule Score Distribution: Clean vs AML")
    ax.legend()

# Alert level by AML
ax = axes[1, 1]
if "alert_level" in df.columns:
    ct = pd.crosstab(df["alert_level"], df["is_aml"], normalize="index") * 100
    ct = ct.reindex(["Critical", "High", "Medium", "Low", "None"])
    ct.plot(kind="barh", stacked=True, ax=ax, color=colors)
    ax.set_title("AML Rate by Alert Level")
    ax.set_xlabel("Percentage")
    ax.legend(["Clean", "AML"])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_initial_eda.png"), bbox_inches="tight")
plt.show()
print(f"\n  Saved: {OUTPUT_DIR}/01_initial_eda.png")



INITIAL EDA

── 3.1: Target Variable (is_aml) ──
  is_aml=0:    204,230 (61.2%)
  is_aml=1:    129,645 (38.8%)
  Imbalance ratio: 1.6:1 (Clean:AML)

── 3.2: Column Types ──
  Numeric: 226 | Object/String: 92 | Boolean: 0

── 3.3: Missing Values (top 20) ──
  No missing values found

── 3.4: Typology Distribution (within is_aml=1) ──
  Charity Abuse                              86,210 (66.5%)
  Funnel Account Network                     47,141 (36.4%)
  Third-Party Payment Web                    21,739 (16.8%)
  Pass-Through Transit Hub                    3,096 (2.4%)
  High-Risk Corridor Transfer                 3,047 (2.4%)
  Structuring (Smurfing)                      2,825 (2.2%)
  Rapid Multi-Hop Layering                    2,547 (2.0%)
  Underground Banking (Hawala)                1,704 (1.3%)
  Money Mule Network                          1,468 (1.1%)
  Circular Transaction Loop                   1,144 (0.9%)

── 3.5: Key Feature Statistics (AML vs Clean) ──

  Feature            

## 4 — Feature Classification
Features are split into 3 tiers:
1. **Protected** (always included): Rule flags + encoded categoricals
2. **Selectable** (subject to feature selection): Graph, velocity, balance, IP features
3. **Excluded**: Labels, IDs, post-hoc scores, typology/convergence/temporal signals


In [4]:
print("=" * 90)
print("FEATURE CLASSIFICATION — 3-Tier Strategy")
print("=" * 90)

# ═══ TIER 3: EXCLUDED — Never used as features ═══

LABEL_COLS = {
    "is_aml", "is_aml_typology", "aml_typology", "typology_group_id", "aml_flag_source"
}

ID_COLS = {
    "transaction_id", "timestamp", "datestamp", "customer_account_number",
    "customer_cif_id", "counterparty_account_number", "customer_name",
    "counterparty_name", "merchant_id", "merchant_name", "merchant_location",
    "session_id", "device_id_fingerprint", "ip_address", "pan", "aadhaar_number",
    "mobile_number", "email_id", "wallet_account_id", "beneficiary_wallet_id_vpa",
    "load_source_account_card_details", "customer_branch_ifsc_code",
    "counterparty_branch_ifsc_swift", "customer_cif_creation_date",
    "kyc_update_date", "account_wallet_opening_date", "account_wallet_inoperative_date",
    "date_of_birth", "date_of_incorporation",
    "father_spouse_name", "identification_proof_doc_no", "entity_identification_proof_doc_no",
    "cif_beneficial_owners", "name_beneficial_owners",
    "address_registered_office", "address_place_of_business",
    "address_beneficial_owners", "address_individual_customer",
    "place_of_incorporation", "browser_app_information",
    "geo_location_city_country", "escrow_account_linked",
    "gps_coordinates_lat", "gps_coordinates_lon",
    "customer_address_lat", "customer_address_lon"
}

# Post-hoc scores derived from is_aml — data leakage
POSTHOC_COLS = {
    "fraud_intensity_score", "fraud_intensity_score_raw", "fis_band",
    "alert_level", "rules_triggered", "rules_triggered_count",
    "predicted_aml", "predicted_typology", "typology_confidence"
}
POSTHOC_COLS.update({c for c in df.columns if c.startswith("prob_")})

# Typology-derived signals — leakage because they're computed using typology labels
TYPOLOGY_LEAKAGE_COLS = {c for c in df.columns if any(c.startswith(p) for p in
    ["typology_signal", "ts_", "convergence_risk", "cr_", "temporal_risk", "tr_"])}
# Also catch exact column names
TYPOLOGY_LEAKAGE_COLS.update({"typology_signal", "convergence_risk", "temporal_risk"})

INTERNAL_COLS = {c for c in df.columns if c.startswith("_")}

all_exclude = LABEL_COLS | ID_COLS | POSTHOC_COLS | TYPOLOGY_LEAKAGE_COLS | INTERNAL_COLS

# ═══ TIER 1: PROTECTED — Always included, no selection applied ═══

# Rule flag columns (binary 0/1 from 126-rule engine)
rule_flags = sorted([c for c in df.columns if c.startswith("rule_") and c not in
              {"rule_score", "rules_triggered", "rules_triggered_count"}
              and c not in all_exclude])

# rule_score itself (composite)
core_scores = [c for c in ["rule_score", "transaction_amount", "annual_income",
               "credit_summation_period", "debit_summation_period",
               "professional_experience_years"] if c in df.columns and c not in all_exclude]

# Categorical features to encode
STRING_FEATURES = sorted([c for c in [
    "transaction_type_dr_cr", "transaction_mode_channel_bank", "cash_flag",
    "transaction_type_ppi", "transaction_mode_channel_ppi", "transaction_status",
    "account_wallet_status", "pep_flag", "hni_flag", "minor_flag",
    "customer_type", "customer_entity_type", "account_category", "account_type",
    "customer_occupation_industry", "vkyc_flag", "wallet_kyc_category",
    "vpn_flag", "emulator_flag", "refund_chargeback_flag",
    "customer_current_risk_score", "tax_residency", "residency",
    "nationality", "citizenship", "non_face_to_face_flag",
    "merchant_category_code", "load_instrument_type", "authentication_method",
    "beneficial_owner_types", "passive_nfe", "source_of_funds",
    "source_of_funds_wallet", "currency"
] if c in df.columns and c not in all_exclude])

PROTECTED_NUMERIC = rule_flags + core_scores
# (encoded categoricals will be added after encoding step)

# ═══ TIER 2: SELECTABLE — Subject to feature selection ═══

SELECTABLE_PREFIXES = [
    "sender_acct_", "sender_cust_", "sender_running", "sender_daily",
    "sender_balance", "sender_pct_", "sender_cumulative",
    "receiver_acct_", "receiver_running", "receiver_balance",
    "inflow_outflow_", "ip_risk", "ip_flag", "ip_txn", "ip_unique", "ip_cross"
]

selectable_features = sorted([c for c in df.select_dtypes(include=[np.number]).columns
                               if c not in all_exclude
                               and c not in PROTECTED_NUMERIC
                               and any(c.startswith(p) for p in SELECTABLE_PREFIXES)])

# ═══ SUMMARY ═══
print(f"\n  ┌─────────────────────────────────────────────────────────────────────────┐")
print(f"  │  TIER 1 — PROTECTED (always included, no selection)                    │")
print(f"  │    Rule flags (binary):          {len(rule_flags):>4d}                                  │")
print(f"  │    Core numeric scores:          {len(core_scores):>4d}                                  │")
print(f"  │    Categorical (to encode):      {len(STRING_FEATURES):>4d}                                  │")
print(f"  ├─────────────────────────────────────────────────────────────────────────┤")
print(f"  │  TIER 2 — SELECTABLE (feature selection applied)                       │")
print(f"  │    Graph/velocity/balance/IP:    {len(selectable_features):>4d}                                  │")
print(f"  ├─────────────────────────────────────────────────────────────────────────┤")
print(f"  │  TIER 3 — EXCLUDED                                                     │")
print(f"  │    Labels:                       {len(LABEL_COLS):>4d}                                  │")
print(f"  │    Identifiers:                  {len(ID_COLS):>4d}                                  │")
print(f"  │    Post-hoc / leakage:           {len(POSTHOC_COLS):>4d}                                  │")
print(f"  │    Typology/convergence/temporal: {len(TYPOLOGY_LEAKAGE_COLS):>4d}                                  │")
print(f"  │    Internal working:             {len(INTERNAL_COLS):>4d}                                  │")
print(f"  └─────────────────────────────────────────────────────────────────────────┘")

# Show excluded typology columns explicitly
print(f"\n  Typology-derived columns EXCLUDED (would cause leakage):")
for c in sorted(TYPOLOGY_LEAKAGE_COLS):
    if c in df.columns:
        print(f"    ✗ {c}")



FEATURE CLASSIFICATION — 3-Tier Strategy

  ┌─────────────────────────────────────────────────────────────────────────┐
  │  TIER 1 — PROTECTED (always included, no selection)                    │
  │    Rule flags (binary):           126                                  │
  │    Core numeric scores:             6                                  │
  │    Categorical (to encode):        34                                  │
  ├─────────────────────────────────────────────────────────────────────────┤
  │  TIER 2 — SELECTABLE (feature selection applied)                       │
  │    Graph/velocity/balance/IP:      78                                  │
  ├─────────────────────────────────────────────────────────────────────────┤
  │  TIER 3 — EXCLUDED                                                     │
  │    Labels:                          5                                  │
  │    Identifiers:                    46                                  │
  │    Post-hoc / leakage:     

## 5 — Encode Categorical Features & Save Mappings


In [5]:
print("Encoding categorical features...")
import pickle

df_ml = df.copy()
encoded_cols = []
encoding_maps = {}

for col in STRING_FEATURES:
    if col not in df_ml.columns:
        continue
    vals = df_ml[col].astype(str).str.strip().str.upper()
    vals = vals.replace({"NAN": "", "NONE": "", "": "MISSING"})
    categories = sorted(vals.unique())
    cat_map = {cat: i for i, cat in enumerate(categories)}
    encoded_col = f"{col}_enc"
    df_ml[encoded_col] = vals.map(cat_map).fillna(-1).astype(int)
    encoded_cols.append(encoded_col)
    encoding_maps[col] = cat_map
    print(f"  {col:<45s} → {encoded_col:<50s} ({len(categories)} categories)")

print(f"\n  Total encoded columns: {len(encoded_cols)}")

# Save encoding maps
import json as _json
json_path = os.path.join(OUTPUT_DIR, "label_encoding_maps.json")
with open(json_path, "w") as f:
    _json.dump({col: {str(k): int(v) for k, v in m.items()} for col, m in encoding_maps.items()}, f, indent=2)
pkl_path = os.path.join(OUTPUT_DIR, "label_encoding_maps.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump(encoding_maps, f)
enc_csv_dir = os.path.join(OUTPUT_DIR, "encoding_csvs")
os.makedirs(enc_csv_dir, exist_ok=True)
for col, mapping in encoding_maps.items():
    pd.DataFrame([{"category": k, "encoded_value": v} for k, v in mapping.items()]).sort_values("encoded_value").to_csv(
        os.path.join(enc_csv_dir, f"{col}_encoding.csv"), index=False)
print(f"  Saved: {json_path}, {pkl_path}, {enc_csv_dir}/ ({len(encoding_maps)} CSVs)")

# Build PROTECTED features (rule flags + core scores + encoded categoricals)
PROTECTED_FEATURES = PROTECTED_NUMERIC + encoded_cols
print(f"\n  PROTECTED features (always in model): {len(PROTECTED_FEATURES)}")
print(f"    Rule flags: {len(rule_flags)} | Core scores: {len(core_scores)} | Encoded cats: {len(encoded_cols)}")



Encoding categorical features...
  account_category                              → account_category_enc                               (7 categories)
  account_type                                  → account_type_enc                                   (5 categories)
  account_wallet_status                         → account_wallet_status_enc                          (2 categories)
  authentication_method                         → authentication_method_enc                          (5 categories)
  beneficial_owner_types                        → beneficial_owner_types_enc                         (4 categories)
  cash_flag                                     → cash_flag_enc                                      (2 categories)
  citizenship                                   → citizenship_enc                                    (15 categories)
  currency                                      → currency_enc                                       (6 categories)
  customer_current_risk_score         

## 6 — Correlation Analysis (Selectable Features Only)


In [6]:
print("=" * 90)
print("CORRELATION ANALYSIS — SELECTABLE Features vs is_aml")
print("(Rule flags + categoricals are PROTECTED and skip this step)")
print("=" * 90)

target = df_ml["is_aml"].astype(float)

# Correlations for SELECTABLE features only
sel_correlations = {}
for feat in selectable_features:
    if feat not in df_ml.columns: continue
    vals = pd.to_numeric(df_ml[feat], errors="coerce").fillna(0)
    if vals.std() == 0:
        sel_correlations[feat] = 0.0
        continue
    sel_correlations[feat] = vals.corr(target)

corr_df = pd.DataFrame([
    {"feature": k, "correlation": v, "abs_correlation": abs(v)}
    for k, v in sel_correlations.items()
]).sort_values("abs_correlation", ascending=False)

# Also compute correlations for PROTECTED features (for reporting only, no selection)
prot_correlations = {}
for feat in PROTECTED_FEATURES:
    if feat not in df_ml.columns: continue
    vals = pd.to_numeric(df_ml[feat], errors="coerce").fillna(0)
    if vals.std() == 0: prot_correlations[feat] = 0.0; continue
    prot_correlations[feat] = vals.corr(target)

prot_corr_df = pd.DataFrame([
    {"feature": k, "correlation": v, "abs_correlation": abs(v)}
    for k, v in prot_correlations.items()
]).sort_values("abs_correlation", ascending=False)

print(f"\n── 6.1: Top 30 SELECTABLE Features by Correlation with is_aml ──")
print(f"  (These are the features subject to selection)\n")
print(f"  {'Rank':<5s} {'Feature':<55s} {'Correlation':>12s} {'Signal':>8s}")
print("  " + "─" * 83)
for i, (_, row) in enumerate(corr_df.head(30).iterrows(), 1):
    strength = "STRONG" if row["abs_correlation"] > 0.1 else ("MEDIUM" if row["abs_correlation"] > 0.05 else "WEAK")
    bar = "█" * int(row["abs_correlation"] * 200)
    print(f"  {i:<5d} {row['feature']:<55s} {row['correlation']:>+11.6f} {strength:<8s} {bar}")

print(f"\n── 6.2: Top 20 PROTECTED Features by Correlation (for reference, NOT selected out) ──\n")
print(f"  {'Rank':<5s} {'Feature':<55s} {'Correlation':>12s} {'Status':>10s}")
print("  " + "─" * 85)
for i, (_, row) in enumerate(prot_corr_df.head(20).iterrows(), 1):
    print(f"  {i:<5d} {row['feature']:<55s} {row['correlation']:>+11.6f} {'PROTECTED':>10s}")

# Heatmap (top selectable + top protected)
top_sel = corr_df.head(15)["feature"].tolist()
top_prot = prot_corr_df.head(5)["feature"].tolist()
heatmap_feats = top_sel + top_prot + ["is_aml"]
heatmap_feats = [c for c in heatmap_feats if c in df_ml.columns]

fig, ax = plt.subplots(figsize=(16, 14))
hm = df_ml[heatmap_feats].corr()
mask = np.triu(np.ones_like(hm, dtype=bool))
sns.heatmap(hm, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            ax=ax, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Top Selectable + Protected Features — Correlation Heatmap", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_correlation_heatmap.png"), bbox_inches="tight")
plt.show()
print(f"\n  Saved: {OUTPUT_DIR}/02_correlation_heatmap.png")
corr_df.to_csv(os.path.join(OUTPUT_DIR, "selectable_feature_correlations.csv"), index=False)



CORRELATION ANALYSIS — SELECTABLE Features vs is_aml
(Rule flags + categoricals are PROTECTED and skip this step)

── 6.1: Top 30 SELECTABLE Features by Correlation with is_aml ──
  (These are the features subject to selection)

  Rank  Feature                                                  Correlation   Signal
  ───────────────────────────────────────────────────────────────────────────────────
  1     sender_acct_txn_count_30d                                 -0.262509 STRONG   ████████████████████████████████████████████████████
  2     sender_acct_outflow_count_30d                             -0.258154 STRONG   ███████████████████████████████████████████████████
  3     sender_cust_outflow_count_30d                             -0.245337 STRONG   █████████████████████████████████████████████████
  4     sender_cust_txn_count_30d                                 -0.244617 STRONG   ████████████████████████████████████████████████
  5     sender_acct_inflow_count_30d                   

## 7 — Multicollinearity & VIF (Selectable Features Only)


In [7]:
print("=" * 90)
print("MULTICOLLINEARITY & VIF — Selectable Features Only")
print("(Rule flags + categoricals are PROTECTED, not checked here)")
print("=" * 90)

from sklearn.linear_model import LinearRegression

THRESHOLD = 0.90
valid_sel = [f for f in selectable_features if f in df_ml.columns
             and pd.to_numeric(df_ml[f], errors="coerce").std() > 0]

print(f"  Computing pairwise correlations for {len(valid_sel)} selectable features...")
if len(df_ml) > 50000:
    sample_df = df_ml[valid_sel].sample(50000, random_state=42)
else:
    sample_df = df_ml[valid_sel].copy()
for c in sample_df.columns:
    sample_df[c] = pd.to_numeric(sample_df[c], errors="coerce").fillna(0)

corr_all = sample_df.corr()

# Find pairs
high_corr_pairs = []
for i in range(len(corr_all.columns)):
    for j in range(i+1, len(corr_all.columns)):
        r = corr_all.iloc[i, j]
        if abs(r) >= THRESHOLD:
            high_corr_pairs.append((corr_all.columns[i], corr_all.columns[j], r))
high_corr_pairs.sort(key=lambda x: -abs(x[2]))

target_corr = {row["feature"]: row["abs_correlation"] for _, row in corr_df.iterrows()}

print(f"\n── 7.1: Highly Correlated Selectable Pairs (|r| >= {THRESHOLD}) ──")
print(f"  Found: {len(high_corr_pairs)} pairs\n")
print(f"  {'Feature A':<45s} {'Feature B':<45s} {'Corr':>8s} {'TgtCorr A':>10s} {'TgtCorr B':>10s} {'Recommendation':>20s}")
print("  " + "─" * 142)
for f1, f2, r in high_corr_pairs[:40]:
    c1 = target_corr.get(f1, 0); c2 = target_corr.get(f2, 0)
    rec = f"Drop {f2[:18]}" if c1 >= c2 else f"Drop {f1[:18]}"
    print(f"  {f1:<45s} {f2:<45s} {r:>+7.4f} {c1:>9.6f} {c2:>9.6f}   {rec}")

# VIF
print(f"\n── 7.2: VIF Scores (Selectable Features) ──")
top_vif = corr_df.head(min(50, len(valid_sel)))["feature"].tolist()
top_vif = [f for f in top_vif if f in sample_df.columns]
print(f"  Computing VIF for {len(top_vif)} features...")

X_vif = sample_df[top_vif].copy()
X_vif = (X_vif - X_vif.mean()) / X_vif.std().replace(0, 1)

vif_results = []
lr = LinearRegression()
for i, feat in enumerate(top_vif):
    y_vif = X_vif[feat].values; X_others = X_vif.drop(columns=[feat]).values
    try:
        lr.fit(X_others, y_vif); r2 = lr.score(X_others, y_vif)
        vif = 1 / max(1 - r2, 0.0001)
    except: vif = float("inf"); r2 = 0
    vif_results.append({"feature": feat, "vif": vif, "r_squared": r2, "target_corr": target_corr.get(feat, 0)})

vif_df = pd.DataFrame(vif_results).sort_values("vif", ascending=False)

print(f"\n  {'Rank':<5s} {'Feature':<50s} {'VIF':>10s} {'R²':>8s} {'Target Corr':>12s} {'Severity':>12s}")
print("  " + "─" * 102)
for i, (_, row) in enumerate(vif_df.iterrows(), 1):
    v = row["vif"]
    sev = "⚠ CRITICAL" if v>=50 else ("⚡ SEVERE" if v>=10 else ("● MODERATE" if v>=5 else "✓ OK"))
    vd = f"{v:>10.2f}" if v < 10000 else f"{v:>10.0f}"
    print(f"  {i:<5d} {row['feature']:<50s} {vd} {row['r_squared']:>7.4f} {row['target_corr']:>11.6f} {sev}")

# Summary
crit=len(vif_df[vif_df["vif"]>=50]); sev=len(vif_df[(vif_df["vif"]>=10)&(vif_df["vif"]<50)])
mod=len(vif_df[(vif_df["vif"]>=5)&(vif_df["vif"]<10)]); ok=len(vif_df[vif_df["vif"]<5])
print(f"\n  VIF Summary: ✓ OK={ok} | ● Moderate={mod} | ⚡ Severe={sev} | ⚠ Critical={crit}")

vif_df.to_csv(os.path.join(OUTPUT_DIR, "vif_scores_selectable.csv"), index=False)

# Greedy removal (on selectable only)
to_remove = set()
for f1, f2, r in high_corr_pairs:
    if f1 in to_remove or f2 in to_remove: continue
    c1 = target_corr.get(f1, 0); c2 = target_corr.get(f2, 0)
    to_remove.add(f2 if c1 >= c2 else f1)

selectable_after_multicollinearity = [f for f in selectable_features if f not in to_remove]
print(f"\n  Selectable before: {len(selectable_features)} → after multicollinearity: {len(selectable_after_multicollinearity)} (removed {len(to_remove)})")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
top30_vif = vif_df.head(30).sort_values("vif")
colors = ["#e74c3c" if v>=50 else "#f39c12" if v>=10 else "#3498db" if v>=5 else "#2ecc71" for v in top30_vif["vif"]]
axes[0].barh(top30_vif["feature"], top30_vif["vif"], color=colors)
axes[0].axvline(x=5, color="orange", linestyle="--", alpha=0.7); axes[0].axvline(x=10, color="red", linestyle="--", alpha=0.7)
axes[0].set_title("VIF Scores (Selectable Features)", fontsize=12, fontweight="bold")
sizes = [ok, mod, sev, crit]; labels = [f"OK<5 ({ok})", f"Mod 5-10 ({mod})", f"Sev 10-50 ({sev})", f"Crit 50+ ({crit})"]
axes[1].pie([s for s in sizes if s>0], labels=[l for l,s in zip(labels,sizes) if s>0],
            colors=["#2ecc71","#3498db","#f39c12","#e74c3c"][:sum(1 for s in sizes if s>0)], autopct="%1.0f%%")
axes[1].set_title("VIF Severity Distribution", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_vif_analysis.png"), bbox_inches="tight")
plt.show()



MULTICOLLINEARITY & VIF — Selectable Features Only
(Rule flags + categoricals are PROTECTED, not checked here)
  Computing pairwise correlations for 77 selectable features...

── 7.1: Highly Correlated Selectable Pairs (|r| >= 0.9) ──
  Found: 10 pairs

  Feature A                                     Feature B                                         Corr  TgtCorr A  TgtCorr B       Recommendation
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
  sender_balance_after_txn                      sender_balance_before_txn                     +0.9871  0.106140  0.112076   Drop sender_balance_aft
  receiver_balance_after_txn                    receiver_balance_before_txn                   +0.9868  0.109956  0.111851   Drop receiver_balance_a
  sender_cust_outflow_count_30d                 sender_cust_txn_count_30d                     +0.9691  0.245337  0.244617   Drop sender_cust_txn_co
  receiver_

## 8 — Feature Importance (Selectable Features — LightGBM Scan)


In [8]:
print("=" * 90)
print("FEATURE IMPORTANCE — Quick LightGBM on SELECTABLE Features Only")
print("=" * 90)

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    print("  LightGBM not installed. Run: pip install lightgbm")
    HAS_LGB = False

if HAS_LGB:
    X_sel = df_ml[selectable_after_multicollinearity].copy()
    for c in X_sel.columns: X_sel[c] = pd.to_numeric(X_sel[c], errors="coerce").fillna(0)
    y_sel = df_ml["is_aml"].astype(int)

    train_ds = lgb.Dataset(X_sel, label=y_sel)
    params = {"objective":"binary","metric":"auc","learning_rate":0.1,"num_leaves":31,
              "max_depth":6,"min_child_samples":50,"subsample":0.8,"colsample_bytree":0.8,
              "scale_pos_weight":(y_sel==0).sum()/max((y_sel==1).sum(),1),
              "verbosity":-1,"random_state":42,"n_jobs":-1}

    print("  Training quick LightGBM for feature importance...")
    model_imp = lgb.train(params, train_ds, num_boost_round=200)

    importance_gain = pd.DataFrame({
        "feature": selectable_after_multicollinearity,
        "gain": model_imp.feature_importance(importance_type="gain"),
        "split": model_imp.feature_importance(importance_type="split")
    }).sort_values("gain", ascending=False)

    print(f"\n── Top 30 Selectable Features by Gain ──")
    print(f"  {'Rank':<5s} {'Feature':<55s} {'Gain':>12s} {'Splits':>8s} {'Cum%':>7s}")
    print("  " + "─" * 90)
    total_gain = importance_gain["gain"].sum()
    cum = 0
    for i, (_, row) in enumerate(importance_gain.head(30).iterrows(), 1):
        pct = row["gain"]/max(total_gain,1)*100; cum += pct
        print(f"  {i:<5d} {row['feature']:<55s} {row['gain']:>12.1f} {row['split']:>8.0f} {cum:>6.1f}%")

    zero_imp = importance_gain[importance_gain["gain"] == 0]
    selected_graph_features = importance_gain[importance_gain["gain"] > 0]["feature"].tolist()
    print(f"\n  Selectable features with importance > 0: {len(selected_graph_features)}")
    print(f"  Removed (zero importance): {len(zero_imp)}")

    fig, ax = plt.subplots(figsize=(14, 10))
    top30 = importance_gain.head(30).sort_values("gain")
    ax.barh(top30["feature"], top30["gain"], color="#3498db")
    ax.set_title("Top 30 Selectable Features by Gain", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "03_feature_importance_selectable.png"), bbox_inches="tight")
    plt.show()

    importance_gain.to_csv(os.path.join(OUTPUT_DIR, "feature_importance_selectable.csv"), index=False)
else:
    selected_graph_features = selectable_after_multicollinearity



FEATURE IMPORTANCE — Quick LightGBM on SELECTABLE Features Only
  Training quick LightGBM for feature importance...

── Top 30 Selectable Features by Gain ──
  Rank  Feature                                                         Gain   Splits    Cum%
  ──────────────────────────────────────────────────────────────────────────────────────────
  1     sender_acct_txn_count_30d                                   129231.2      266   15.9%
  2     sender_running_balance_txn_amount                           125321.4      690   31.4%
  3     sender_balance_before_txn                                    78304.7      427   41.0%
  4     receiver_balance_before_txn                                  77720.1      420   50.6%
  5     sender_cust_outflow_count_30d                                47325.2      183   56.4%
  6     receiver_acct_outflow_count_30d                              46598.1      213   62.2%
  7     sender_acct_outflow_amt_30d                                  39983.8      307   67.

## 9 — Feature Selection Summary & Final Feature Set


In [9]:
print("=" * 90)
print("FEATURE SELECTION SUMMARY")
print("=" * 90)

# Final features = PROTECTED (all retained) + SELECTED graph features
features_final = PROTECTED_FEATURES + selected_graph_features
features_final = list(dict.fromkeys(features_final))  # deduplicate

print(f"""
  ┌──────────────────────────────────────────────────────────────────────────────┐
  │  FINAL FEATURE SET                                                          │
  ├──────────────────────────────────────────────────────────────────────────────┤
  │  PROTECTED (no selection, always included):                                 │
  │    Rule flags:                   {len(rule_flags):>4d}  (126 binary rule columns)          │
  │    Encoded categoricals:         {len(encoded_cols):>4d}  (occupation, channel, risk, etc)   │
  │    Core numeric scores:          {len(core_scores):>4d}  (rule_score, amount, income, etc)   │
  │    Subtotal PROTECTED:           {len(PROTECTED_FEATURES):>4d}                                       │
  ├──────────────────────────────────────────────────────────────────────────────┤
  │  SELECTED (after correlation + VIF + importance):                           │
  │    Started with:                 {len(selectable_features):>4d}  graph/velocity/balance/IP features│
  │    After multicollinearity:      {len(selectable_after_multicollinearity):>4d}  (removed {len(to_remove)} correlated pairs)      │
  │    After zero-importance:        {len(selected_graph_features):>4d}  (final selected)                  │
  ├──────────────────────────────────────────────────────────────────────────────┤
  │  EXCLUDED:                                                                  │
  │    Typology/convergence/temporal: {len(TYPOLOGY_LEAKAGE_COLS):>4d}  (leakage from typology labels)    │
  │    FIS/alert_level/fis_band:     {len(POSTHOC_COLS):>4d}  (post-hoc derived scores)          │
  │    Labels + IDs:                 {len(LABEL_COLS)+len(ID_COLS):>4d}  (target + identifiers)             │
  ├──────────────────────────────────────────────────────────────────────────────┤
  │  TOTAL FEATURES IN MODEL:       {len(features_final):>4d}                                       │
  └──────────────────────────────────────────────────────────────────────────────┘
""")

# Category breakdown
categories = {
    "Rule flags (126 binary)": [f for f in features_final if f.startswith("rule_") and f != "rule_score"],
    "Core scores (rule_score, amount, income)": [f for f in features_final if f in core_scores],
    "Encoded categoricals": [f for f in features_final if f.endswith("_enc")],
    "Sender account velocity": [f for f in features_final if f.startswith("sender_acct_")],
    "Sender customer velocity": [f for f in features_final if f.startswith("sender_cust_")],
    "Sender balance": [f for f in features_final if "sender" in f and any(k in f for k in ["balance","running","cumulative","pct_balance","daily"])],
    "Receiver features": [f for f in features_final if f.startswith("receiver_")],
    "IP risk features": [f for f in features_final if f.startswith("ip_")],
    "Volume ratios": [f for f in features_final if "volume_balance_ratio" in f],
}
print(f"  Breakdown:")
for cat, cols in categories.items():
    if cols: print(f"    {cat:<45s} {len(cols):>4d}")

with open(os.path.join(OUTPUT_DIR, "selected_features.txt"), "w") as f:
    for feat in features_final: f.write(feat + "\n")
print(f"\n  Saved: {OUTPUT_DIR}/selected_features.txt ({len(features_final)} features)")



FEATURE SELECTION SUMMARY

  ┌──────────────────────────────────────────────────────────────────────────────┐
  │  FINAL FEATURE SET                                                          │
  ├──────────────────────────────────────────────────────────────────────────────┤
  │  PROTECTED (no selection, always included):                                 │
  │    Rule flags:                    126  (126 binary rule columns)          │
  │    Encoded categoricals:           34  (occupation, channel, risk, etc)   │
  │    Core numeric scores:             6  (rule_score, amount, income, etc)   │
  │    Subtotal PROTECTED:            166                                       │
  ├──────────────────────────────────────────────────────────────────────────────┤
  │  SELECTED (after correlation + VIF + importance):                           │
  │    Started with:                   78  graph/velocity/balance/IP features│
  │    After multicollinearity:        70  (removed 8 correlated pairs)     

## 10 — Class Imbalance Handling
Three strategies compared:
1. **Class weights** (`scale_pos_weight` in LightGBM) — penalizes misclassifying minority class
2. **SMOTE** (Synthetic Minority Oversampling) — generates synthetic AML examples
3. **Threshold tuning** — adjusts decision boundary instead of 0.5


In [10]:
print("=" * 90)
print("CLASS IMBALANCE ANALYSIS & HANDLING")
print("=" * 90)

from sklearn.model_selection import train_test_split

# Prepare data
X = df_ml[features_final].copy()
for c in X.columns: X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)
y = df_ml["is_aml"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()
imbalance_ratio = n_neg / max(n_pos, 1)

print(f"\n  Training set class distribution:")
print(f"    Clean (0): {n_neg:>10,} ({n_neg/len(y_train)*100:.1f}%)")
print(f"    AML   (1): {n_pos:>10,} ({n_pos/len(y_train)*100:.1f}%)")
print(f"    Imbalance ratio: {imbalance_ratio:.2f}:1")

# ── Strategy 1: Class Weights ──
print(f"\n── Strategy 1: Class Weights (scale_pos_weight = {imbalance_ratio:.2f}) ──")
print(f"  LightGBM multiplies the loss for AML class by {imbalance_ratio:.2f}x")
print(f"  Effect: Model penalized {imbalance_ratio:.1f}x more for missing an AML transaction")

# ── Strategy 2: SMOTE ──
print(f"\n── Strategy 2: SMOTE (Synthetic Minority Oversampling) ──")
try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False
    print("  imblearn not installed. Run: pip install imbalanced-learn")

if HAS_SMOTE:
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    print(f"  Before SMOTE: {len(X_train):,} rows (AML={n_pos:,})")
    print(f"  After SMOTE:  {len(X_train_smote):,} rows (AML={(y_train_smote==1).sum():,})")
    print(f"  Synthetic AML samples added: {(y_train_smote==1).sum() - n_pos:,}")
else:
    X_train_smote = X_train; y_train_smote = y_train

# ── Strategy 3: Threshold Tuning ──
print(f"\n── Strategy 3: Threshold Tuning ──")
print(f"  Default threshold: 0.5 (equal weight to precision and recall)")
print(f"  For AML: lower threshold (e.g., 0.3) increases recall at cost of precision")
print(f"  Will test thresholds: 0.3, 0.4, 0.5, 0.6, 0.7 after model training")

# ── Compare all 3 strategies ──
if HAS_LGB:
    from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

    results = {}
    
    configs = {
        "Baseline (no handling)": {
            "X": X_train, "y": y_train,
            "params": {"scale_pos_weight": 1.0}
        },
        "Class Weights": {
            "X": X_train, "y": y_train,
            "params": {"scale_pos_weight": imbalance_ratio}
        },
        "SMOTE": {
            "X": X_train_smote, "y": y_train_smote,
            "params": {"scale_pos_weight": 1.0}
        },
        "Weights + SMOTE": {
            "X": X_train_smote, "y": y_train_smote,
            "params": {"scale_pos_weight": imbalance_ratio * 0.5}  # reduced since SMOTE already balances
        },
    }
    
    print(f"\n── Comparing Imbalance Strategies ──\n")
    print(f"  {'Strategy':<25s} │ {'AUC':>7s} {'F1':>7s} {'Prec':>7s} {'Recall':>7s} │ {'TP':>8s} {'FP':>8s} {'FN':>8s} {'TN':>8s}")
    print("  " + "─" * 95)
    
    for name, cfg in configs.items():
        base_params = {
            "objective":"binary","metric":"auc","learning_rate":0.05,
            "num_leaves":63,"max_depth":8,"min_child_samples":50,
            "subsample":0.8,"colsample_bytree":0.8,
            "verbosity":-1,"random_state":42,"n_jobs":-1
        }
        base_params.update(cfg["params"])
        
        ds = lgb.Dataset(cfg["X"], label=cfg["y"])
        val_ds = lgb.Dataset(X_test, label=y_test, reference=ds)
        
        model = lgb.train(base_params, ds, num_boost_round=300,
                          valid_sets=[val_ds], callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)])
        
        y_prob = model.predict(X_test)
        y_pred = (y_prob >= 0.5).astype(int)
        
        auc = roc_auc_score(y_test, y_prob)
        f1 = f1_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y_test, y_pred)
        tp = cm[1,1]; fp = cm[0,1]; fn = cm[1,0]; tn = cm[0,0]
        
        results[name] = {"auc": auc, "f1": f1, "precision": prec, "recall": rec,
                         "model": model, "y_prob": y_prob, "tp": tp, "fp": fp, "fn": fn, "tn": tn}
        
        print(f"  {name:<25s} │ {auc:>6.4f} {f1:>6.4f} {prec:>6.4f} {rec:>6.4f} │ {tp:>8,} {fp:>8,} {fn:>8,} {tn:>8,}")
    
    # Find best strategy
    best_strategy = max(results, key=lambda k: results[k]["f1"])
    print(f"\n  ► Best strategy by F1: {best_strategy} (F1={results[best_strategy]['f1']:.4f})")
    
    # ── Threshold tuning on best model ──
    best_prob = results[best_strategy]["y_prob"]
    
    print(f"\n── Threshold Tuning on {best_strategy} ──\n")
    print(f"  {'Threshold':>10s} │ {'F1':>7s} {'Prec':>7s} {'Recall':>7s} │ {'TP':>8s} {'FP':>8s} {'FN':>8s}")
    print("  " + "─" * 65)
    
    best_f1 = 0; best_thresh = 0.5
    for thresh in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]:
        yp = (best_prob >= thresh).astype(int)
        f1 = f1_score(y_test, yp); p = precision_score(y_test, yp, zero_division=0); r = recall_score(y_test, yp)
        cm = confusion_matrix(y_test, yp)
        marker = " ◄" if f1 > best_f1 else ""
        if f1 > best_f1: best_f1 = f1; best_thresh = thresh
        print(f"  {thresh:>10.2f} │ {f1:>6.4f} {p:>6.4f} {r:>6.4f} │ {cm[1,1]:>8,} {cm[0,1]:>8,} {cm[1,0]:>8,}{marker}")
    
    print(f"\n  ► Optimal threshold: {best_thresh} (F1={best_f1:.4f})")
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Strategy comparison
    strats = list(results.keys())
    metrics = ["auc", "f1", "precision", "recall"]
    x = np.arange(len(strats)); width = 0.2
    for i, m in enumerate(metrics):
        vals = [results[s][m] for s in strats]
        axes[0].bar(x + i*width, vals, width, label=m.upper())
    axes[0].set_xticks(x + width*1.5); axes[0].set_xticklabels(strats, rotation=20, ha="right")
    axes[0].set_title("Imbalance Strategy Comparison", fontweight="bold"); axes[0].legend(); axes[0].set_ylim(0, 1.1)
    
    # Threshold curve
    thresholds = np.arange(0.1, 0.9, 0.02)
    f1s = [f1_score(y_test, (best_prob>=t).astype(int)) for t in thresholds]
    precs = [precision_score(y_test, (best_prob>=t).astype(int), zero_division=0) for t in thresholds]
    recs = [recall_score(y_test, (best_prob>=t).astype(int)) for t in thresholds]
    axes[1].plot(thresholds, f1s, "b-", lw=2, label="F1"); axes[1].plot(thresholds, precs, "g--", label="Precision")
    axes[1].plot(thresholds, recs, "r--", label="Recall"); axes[1].axvline(best_thresh, color="k", linestyle=":", alpha=0.5)
    axes[1].set_title("Threshold Tuning Curve", fontweight="bold"); axes[1].set_xlabel("Threshold"); axes[1].legend()
    
    # Score distribution
    axes[2].hist(best_prob[y_test==0], bins=50, alpha=0.6, label="Clean", color="#2ecc71", density=True)
    axes[2].hist(best_prob[y_test==1], bins=50, alpha=0.6, label="AML", color="#e74c3c", density=True)
    axes[2].axvline(best_thresh, color="k", linestyle="--", label=f"Threshold={best_thresh}")
    axes[2].set_title("Score Distribution", fontweight="bold"); axes[2].legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "06_imbalance_handling.png"), bbox_inches="tight")
    plt.show()
    print(f"\n  Saved: {OUTPUT_DIR}/06_imbalance_handling.png")



CLASS IMBALANCE ANALYSIS & HANDLING

  Training set class distribution:
    Clean (0):    163,384 (61.2%)
    AML   (1):    103,716 (38.8%)
    Imbalance ratio: 1.58:1

── Strategy 1: Class Weights (scale_pos_weight = 1.58) ──
  LightGBM multiplies the loss for AML class by 1.58x
  Effect: Model penalized 1.6x more for missing an AML transaction

── Strategy 2: SMOTE (Synthetic Minority Oversampling) ──
  imblearn not installed. Run: pip install imbalanced-learn

── Strategy 3: Threshold Tuning ──
  Default threshold: 0.5 (equal weight to precision and recall)
  For AML: lower threshold (e.g., 0.3) increases recall at cost of precision
  Will test thresholds: 0.3, 0.4, 0.5, 0.6, 0.7 after model training

── Comparing Imbalance Strategies ──

  Strategy                  │     AUC      F1    Prec  Recall │       TP       FP       FN       TN
  ───────────────────────────────────────────────────────────────────────────────────────────────
Training until validation scores don't improve for

## 11 — Final Baseline Model (Best Imbalance Strategy)


In [11]:
print("=" * 90)
print(f"FINAL BASELINE MODEL — {best_strategy} + Threshold={best_thresh}")
print("=" * 90)

if HAS_LGB:
    from sklearn.metrics import classification_report, roc_curve, precision_recall_curve, average_precision_score
    
    final_model = results[best_strategy]["model"]
    y_prob_final = results[best_strategy]["y_prob"]
    y_pred_final = (y_prob_final >= best_thresh).astype(int)
    
    auc = roc_auc_score(y_test, y_prob_final)
    avg_prec = average_precision_score(y_test, y_prob_final)
    
    print(f"\n  AUC-ROC:           {auc:.4f}")
    print(f"  Average Precision: {avg_prec:.4f}")
    print(f"  Threshold:         {best_thresh}")
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred_final, target_names=["Clean", "AML"], digits=4))
    
    cm = confusion_matrix(y_test, y_pred_final)
    print(f"  Confusion Matrix:")
    print(f"    {'':>15s} {'Pred Clean':>12s} {'Pred AML':>12s}")
    print(f"    {'Actual Clean':<15s} {cm[0,0]:>12,} {cm[0,1]:>12,}")
    print(f"    {'Actual AML':<15s} {cm[1,0]:>12,} {cm[1,1]:>12,}")
    
    # Per-typology recall
    if "aml_typology" in df.columns:
        print(f"\n  ── Per-Typology Detection Rate ──")
        test_df = df.iloc[X_test.index].copy()
        test_df["_pred_prob"] = y_prob_final
        test_df["_pred"] = y_pred_final
        typ_col = "aml_typology"
        all_typs = set()
        for t in test_df[typ_col].dropna():
            for part in str(t).split("; "):
                if part.strip(): all_typs.add(part.strip())
        
        print(f"    {'Typology':<40s} {'Total':>7s} {'Caught':>7s} {'Recall':>7s} {'Avg Prob':>9s} {'Status':>8s}")
        print(f"    {'─'*83}")
        for typ in sorted(all_typs):
            mask = test_df[typ_col].astype(str).str.contains(typ, na=False)
            cnt = mask.sum()
            if cnt == 0: continue
            caught = test_df.loc[mask, "_pred"].sum()
            recall = caught / cnt * 100
            avg_p = test_df.loc[mask, "_pred_prob"].mean()
            status = "✓ GOOD" if recall > 80 else ("⚡ CHECK" if recall > 50 else "⚠ LOW")
            print(f"    {typ:<40s} {cnt:>7,} {caught:>7,} {recall:>6.1f}% {avg_p:>8.3f} {status}")
    
    # Feature importance
    imp = pd.DataFrame({"feature": features_final,
                         "gain": final_model.feature_importance(importance_type="gain")}).sort_values("gain", ascending=False)
    
    print(f"\n  ── Top 20 Features by Importance ──")
    print(f"    {'Rank':<5s} {'Feature':<55s} {'Gain':>12s} {'Category':>15s}")
    print(f"    {'─'*90}")
    for i, (_, row) in enumerate(imp.head(20).iterrows(), 1):
        cat = "RULE" if row["feature"].startswith("rule_") else ("ENCODED" if row["feature"].endswith("_enc") else "GRAPH/VEL")
        print(f"    {i:<5d} {row['feature']:<55s} {row['gain']:>12.1f} {cat:>15s}")
    
    # Plots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f"Final Model Results — {best_strategy} (threshold={best_thresh})", fontsize=14, fontweight="bold")
    
    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob_final)
    axes[0,0].plot(fpr, tpr, "b-", lw=2, label=f"AUC={auc:.4f}")
    axes[0,0].plot([0,1],[0,1],"k--",alpha=0.3); axes[0,0].set_title("ROC Curve"); axes[0,0].legend()
    
    # PR
    prec_c, rec_c, _ = precision_recall_curve(y_test, y_prob_final)
    axes[0,1].plot(rec_c, prec_c, "r-", lw=2, label=f"AP={avg_prec:.4f}")
    axes[0,1].set_title("Precision-Recall Curve"); axes[0,1].legend()
    
    # Confusion matrix heatmap
    sns.heatmap(cm, annot=True, fmt=",", cmap="Blues", ax=axes[1,0],
                xticklabels=["Pred Clean","Pred AML"], yticklabels=["Actual Clean","Actual AML"])
    axes[1,0].set_title("Confusion Matrix")
    
    # Top 15 importance
    top15 = imp.head(15).sort_values("gain")
    colors = ["#e74c3c" if f.startswith("rule_") else "#3498db" if f.endswith("_enc") else "#2ecc71" for f in top15["feature"]]
    axes[1,1].barh(top15["feature"], top15["gain"], color=colors)
    axes[1,1].set_title("Top 15 Feature Importance (Red=Rules, Blue=Categorical, Green=Graph)")
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "07_final_model_results.png"), bbox_inches="tight")
    plt.show()
    
    # Save model + metadata
    final_model.save_model(os.path.join(OUTPUT_DIR, "final_lgb_model.txt"))
    
    X_train.to_parquet(os.path.join(OUTPUT_DIR, "X_train.parquet"), index=False)
    X_test.to_parquet(os.path.join(OUTPUT_DIR, "X_test.parquet"), index=False)
    y_train.to_frame().to_parquet(os.path.join(OUTPUT_DIR, "y_train.parquet"), index=False)
    y_test.to_frame().to_parquet(os.path.join(OUTPUT_DIR, "y_test.parquet"), index=False)
    
    import json as _json
    metadata = {
        "features": features_final,
        "n_features": len(features_final),
        "protected_count": len(PROTECTED_FEATURES),
        "selected_graph_count": len(selected_graph_features),
        "best_strategy": best_strategy,
        "optimal_threshold": best_thresh,
        "auc_roc": auc,
        "f1_score": float(f1_score(y_test, y_pred_final)),
        "imbalance_ratio": imbalance_ratio,
        "n_train": len(X_train),
        "n_test": len(X_test),
    }
    with open(os.path.join(OUTPUT_DIR, "model_metadata.json"), "w") as f:
        _json.dump(metadata, f, indent=2)
    
    print(f"\n  Saved outputs:")
    for fn in sorted(os.listdir(OUTPUT_DIR)):
        if not os.path.isdir(os.path.join(OUTPUT_DIR, fn)):
            size = os.path.getsize(os.path.join(OUTPUT_DIR, fn)) / (1024*1024)
            print(f"    {fn:<45s} {size:>8.2f} MB")

print(f"\n{'='*90}")
print("ML PREPARATION COMPLETE")
print(f"{'='*90}")



FINAL BASELINE MODEL — Class Weights + Threshold=0.45

  AUC-ROC:           0.8659
  Average Precision: 0.8282
  Threshold:         0.45

  Classification Report:
              precision    recall  f1-score   support

       Clean     0.8557    0.7637    0.8071     40846
         AML     0.6817    0.7971    0.7349     25929

    accuracy                         0.7767     66775
   macro avg     0.7687    0.7804    0.7710     66775
weighted avg     0.7881    0.7767    0.7790     66775

  Confusion Matrix:
                      Pred Clean     Pred AML
    Actual Clean          31,194        9,652
    Actual AML             5,261       20,668

  ── Per-Typology Detection Rate ──
    Typology                                   Total  Caught  Recall  Avg Prob   Status
    ───────────────────────────────────────────────────────────────────────────────────
    Charity Abuse                             17,239  15,625   90.6%    0.762 ✓ GOOD
    Circular Transaction Loop                    233  

In [4]:
import pandas as pd
x_train_read = pd.read_parquet(r"C:\Users\VISHNUPRIYA\OneDrive\Desktop\Freelancing\AIGEN\smartsentry_aml_model\python_scripts\ml_outputs\X_train.parquet")
x_train_read.head(2000).to_excel("Check_transactions.xlsx", index=False)